### Metrics

| Metric               | Simple question                           |
| -------------------- | ----------------------------------------- |
| **Hit Rate**         | Did we find the correct document?         |
| **Precision@3**      | How many retrieved chunks are relevant?   |
| **Recall@3**         | Did we retrieve the relevant information? |
| **Faithfulness**     | Is the answer supported by the context?   |
| **Answer Relevance** | Does the answer, answer the question?      |


In [2]:
import ollama
import os

import faiss
import numpy as np

In [3]:

def load_documents(data_dir):

    documents = []

    for file_name in os.listdir(data_dir):

        if file_name.endswith(".txt"):

            file_path = os.path.join( data_dir, file_name )

            with open(file_path, "r", encoding="utf-8" ) as file:

                text = file.read()

            documents.append({ "source": file_name, "text": text })

            print( "Document loaded:", file_name)

    return documents


In [5]:
documents = load_documents("Agriculture")

print( "\nTotal documents:",len(documents))

Document loaded: crop_production.txt
Document loaded: government_schemes.txt
Document loaded: irrigation_management.txt
Document loaded: pest_and_disease.txt
Document loaded: soil_health.txt

Total documents: 5


In [6]:
def chunk_documents(documents):

    chunk_size = 800
    chunk_overlap = 100

    chunks = []

    for document in documents:

        text = document["text"]
        source = document["source"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk = text[start:end]

            chunks.append({ "source": source, "text": chunk})

            start = end - chunk_overlap

    return chunks


In [7]:
chunks = chunk_documents(documents)

print("\nTotal chunks:",len(chunks))


Total chunks: 21


In [31]:
test_questions = [
    {
        "question": "What government schemes are available for farmers?",
        "expected_source": "government_schemes.txt"
    },
    {
        "question": "How should irrigation be managed?",
        "expected_source": "irrigation_management.txt"
    },
    {
        "question": "What are the common signs of pest and disease problems?",
        "expected_source": "pest_and_disease.txt"
    }
]

In [8]:

def create_embeddings(chunks):

    embeddings = []

    for chunk in chunks:

        response = ollama.embeddings(
            model="nomic-embed-text",
            prompt=chunk["text"]
        )

        embeddings.append(
            response["embedding"]
        )

    return np.array(
        embeddings,
        dtype="float32"
    )


In [9]:
chunk_embeddings = create_embeddings(chunks)

print( "\nEmbedding shape:",chunk_embeddings.shape)


Embedding shape: (21, 768)


In [10]:

def create_faiss_index(embeddings):

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2( dimension)
    index.add(embeddings)
    return index

In [11]:

index = create_faiss_index(chunk_embeddings)

print("Vectors stored in FAISS:",index.ntotal)

Vectors stored in FAISS: 21


In [32]:

question = input("\nEnter your question: ")

print( "\nQuestion:",question)


Enter your question:  What government schemes are available for farmers?



Question: What government schemes are available for farmers?


In [33]:

def embed_question(question):
    response = ollama.embeddings(
        model="nomic-embed-text",
        prompt=question
    )

    question_embedding = np.array(
        [response["embedding"]],
        dtype="float32"
    )

    return question_embedding


In [34]:
question_embedding = embed_question(question)

print("\nQuestion embedding shape:",question_embedding.shape)


Question embedding shape: (1, 768)


In [35]:

def search_faiss( question_embedding, index, k=3):

    distances, indices = index.search( question_embedding, k)

    return distances[0], indices[0]


In [36]:

distances, indices = search_faiss( question_embedding, index, k=3)

print( "\nDistances:", distances)
print( "Chunk numbers:", indices)


Distances: [211.29936 304.9206  310.77386]
Chunk numbers: [6 7 8]


In [37]:

def retrieve_chunks( indices, chunks):

    retrieved_chunks = []

    for i in indices:
        retrieved_chunks.append( chunks[i] )
    return retrieved_chunks


In [38]:
retrieved_chunks = retrieve_chunks(indices, chunks)

print( "\nNumber of retrieved chunks:", len(retrieved_chunks))


Number of retrieved chunks: 3


In [39]:

for i, chunk in enumerate( retrieved_chunks):

    print( f"\n===== RETRIEVED CHUNK {i + 1} =====" )

    print( "Source:", chunk["source"])

    print( "Distance:", distances[i])

    print("\nText:")

    print(chunk["text"] )


===== RETRIEVED CHUNK 1 =====
Source: government_schemes.txt
Distance: 211.29936

Text:
AGRICULTURAL GOVERNMENT SCHEMES AND SUBSIDIES DIRECTORY

1. DIRECT INCOME SUPPORT AND FORMAL INSTITUTIONAL CREDIT
- PM-KISAN (Pradhan Mantri Kisan Samman Nidhi): A central sector scheme providing income support of financial aid directly credited to the Aadhaar-seeded bank accounts of eligible landholding farmer families across three equal installments of financial allocations per fiscal cycle.
- Kisan Credit Card (KCC): Institutional credit mechanism empowering farmers to access short-term formal loans at highly subsidized interest rates (often further reduced via prompt repayment interest subventions) to cover working capital requirements for seeds, fertilizers, pesticides, and post-harvest maintenance.

2. RISK MITIGATION AND COMPREHENSIVE CROP INSURANCE
- PMFBY (Pradhan Mantri Fasal Bim

===== RETRIEVED CHUNK 2 =====
Source: government_schemes.txt
Distance: 304.9206

Text:
 maintenance.

2. RISK

In [40]:

def create_context( retrieved_chunks ):

    context = ""

    for chunk in retrieved_chunks:

        context += f"""
Source: {chunk["source"]}

{chunk["text"]}

"""

    return context


In [41]:

context = create_context( retrieved_chunks)

print("\n===== CONTEXT =====")
print(context)



===== CONTEXT =====

Source: government_schemes.txt

AGRICULTURAL GOVERNMENT SCHEMES AND SUBSIDIES DIRECTORY

1. DIRECT INCOME SUPPORT AND FORMAL INSTITUTIONAL CREDIT
- PM-KISAN (Pradhan Mantri Kisan Samman Nidhi): A central sector scheme providing income support of financial aid directly credited to the Aadhaar-seeded bank accounts of eligible landholding farmer families across three equal installments of financial allocations per fiscal cycle.
- Kisan Credit Card (KCC): Institutional credit mechanism empowering farmers to access short-term formal loans at highly subsidized interest rates (often further reduced via prompt repayment interest subventions) to cover working capital requirements for seeds, fertilizers, pesticides, and post-harvest maintenance.

2. RISK MITIGATION AND COMPREHENSIVE CROP INSURANCE
- PMFBY (Pradhan Mantri Fasal Bim


Source: government_schemes.txt

 maintenance.

2. RISK MITIGATION AND COMPREHENSIVE CROP INSURANCE
- PMFBY (Pradhan Mantri Fasal Bima Yojana): 

In [42]:

def create_prompt( question, context ):

    prompt = f"""
You are an agricultural information assistant.

Answer the question using ONLY the information
provided in the context.

Do not use outside knowledge.

If the answer is not available in the context,
say:

"I could not find the answer in the provided documents."

Give a clear and concise answer.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt


In [43]:

prompt = create_prompt( question, context)

In [44]:

def generate_answer(prompt):

    response = ollama.chat(
 
        model="llama3.2",

        messages=[
            { "role": "user", "content": prompt }
        ],

        options={ "temperature": 0}
    )

    return response["message"]["content"]


In [45]:

answer = generate_answer( prompt)

In [46]:

print( "FINAL RAG ANSWER")

print(answer)


FINAL RAG ANSWER
The following government schemes are available for farmers:

1. PM-KISAN (Pradhan Mantri Kisan Samman Nidhi): Provides income support through direct financial aid to eligible landholding farmer families.
2. Kisan Credit Card (KCC): Empowers farmers to access short-term formal loans at highly subsidized interest rates.
3. PMFBY (Pradhan Mantri Fasal Bima Yojana): Protects farmers against yield losses due to natural risks.
4. PM-KUSUM (Pradhan Mantri Kisan Urja Suraksha evam Utthaan Mahabhiyan): Facilitates the installation of standalone off-grid solar agricultural water pumps and deployment of small-scale solar power plants.
5. PKVY (Paramparagat Krishi Vikas Yojana): Promotes chemical-free organic farming clusters through community adoption.

These schemes aim to provide financial support, risk mitigation, and infrastructure development to farmers.


In [47]:

print( "\nSources used:")

for chunk in retrieved_chunks:

    print( "-", chunk["source"] )


Sources used:
- government_schemes.txt
- government_schemes.txt
- government_schemes.txt


### Hit Rate

In [48]:

expected_source = "government_schemes.txt"

### Hit Rate → Did retrieval find the right information?

In [49]:
sources = []

for chunk in retrieved_chunks:
    sources.append(chunk["source"])

if expected_source in sources:
    hit_rate = 1
else:
    hit_rate = 0

print("Hit Rate:", hit_rate)

Hit Rate: 1


### Faithfulness

### Faithfulness → Did the LLM stay within the retrieved information?

This is particularly important in RAG because it helps detect hallucination.

In [50]:
faithfulness_prompt = f"""
Is the following answer completely supported by the context?

Context:
{context}

Answer:
{answer}

Return only 1 or 0.

1 = Supported
0 = Not supported
"""

In [51]:
response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": faithfulness_prompt
        }
    ],
    options={"temperature": 0}
)

In [52]:
faithfulness = response["message"]["content"].strip()

print("Faithfulness:", faithfulness)

Faithfulness: The answer is 1, as the provided answer is completely supported by the context. The answer accurately lists the five government schemes mentioned in the context, and each scheme is described in detail, providing a clear understanding of what each scheme offers to farmers.


#### Answer Relevance


### Answer Relevance → Did the LLM answer the question asked?

In [53]:
relevance_prompt = f"""
Does the answer directly answer the question?

Question:
{question}

Answer:
{answer}

Return only 1 or 0.

1 = Yes
0 = No
"""

In [54]:
response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": relevance_prompt
        }
    ],
    options={"temperature": 0}
)

In [55]:
answer_relevance = response["message"]["content"].strip()

print("Answer Relevance:", answer_relevance)

Answer Relevance: 1


### Precision@3

In [ ]:
#  PRECISION@3

relevant = 0

for chunk in retrieved_chunks:

    if chunk["source"] == expected_source:
        relevant += 1

precision_at_3 = relevant / len(retrieved_chunks)

print("Precision@3      :", round(precision_at_3, 2))

### Recall@3


In [ ]:

Question it answers:

Did we retrieve the relevant information in the Top-3?

Suppose the relevant source is:

government_schemes.txt

    Retrieved:
    
    1. irrigation_management.txt
    2. government_schemes.txt       ✓
    3. pest_and_disease.txt

The relevant source was retrieved.

Therefore:

Recall@3 = 1


Recall = Relevant information retrieved
         -------------------------------
         Total relevant information

In [56]:
# RECALL@3

if expected_source in sources:
    recall_at_3 = 1
else:
    recall_at_3 = 0

print("Recall@3         :", recall_at_3)

Recall@3         : 1


In [ ]:
questions = [
    "What is the optimal soil pH for maximum nutrient availability?",
    "What happens when soil pH is below 5.5?",
    "What is the recommended target level of soil organic carbon?",
    "What are the symptoms of nitrogen deficiency?",
    "What is the role of phosphorus in crops?",
    "What is the role of potassium in plants?",
    "What is Integrated Nutrient Management?",
    "What is the recommended seed germination rate?"
]